### Importation des librairies et du dataset

In [5]:
import numpy as np
import pandas as pd
df = pd.read_csv("C:/Users/hp/Downloads/HealthConnect_Appointment_Data.csv")


In [6]:
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [8]:
df.shape

(5000, 18)

Le jeu contient 5 000 lignes × 18 colonnes. Une ligne = un rendez-vous (un même patient peut apparaître plusieurs fois). Les variables se répartissent en trois familles : patient (gender, age, distance…), rendez-vous (type, dates, créneau, rappel…) et historique/résultat (previous_no_shows, appointment_outcome). La variable cible est appointment_outcome. 

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 non-null   object 
 15  dist

In [10]:
df.head(10)

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show
5,HC-00006,P-0523,Male,66,65+,Specialist Consultation,2/16/2026,4/14/2026,Tuesday,Morning,57,4,2,No,NaN,3.1,36.0,No-Show
6,HC-00007,P-0776,Male,58,55-64,Diagnostic Test,1/17/2025,2/16/2025,Sunday,Afternoon,30,4,1,Yes,WhatsApp,8.2,30.0,No-Show
7,HC-00008,P-0927,Female,28,25-34,Specialist Consultation,11/27/2025,1/16/2026,Friday,Afternoon,50,6,1,Yes,SMS,19.2,21.0,Attended
8,HC-00009,P-0388,Male,77,65+,Follow-up,3/9/2025,4/21/2025,Monday,Afternoon,43,3,0,Yes,SMS,10.1,27.0,No-Show
9,HC-00010,P-0371,Female,20,18-24,Follow-up,2/5/2025,2/21/2025,Friday,Afternoon,16,2,0,Yes,SMS,28.0,20.0,Attended


Aucune colonne vide, mais 12 colonnes en « object », dont les deux dates qui devront être converties. waiting_time_minutes est en float alors que le dictionnaire annonce un entier : signe qu'elle contient des manquants.

In [14]:
df.isnull().sum()

appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

Trois colonnes ont des manquants. reminder_channel (1 366) = manquant logique : sans rappel envoyé, pas de canal (1 366 = le nombre de « reminder_sent = No »). distance (90) et waiting_time (60) = vrais manquants, annoncés par le dictionnaire → décision de traitement à prendre plus tard.

In [21]:
df.dtypes

appointment_id            object
patient_id                object
gender                    object
age                        int64
age_group                 object
appointment_type          object
booking_date              object
appointment_date          object
appointment_day           object
appointment_time          object
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent             object
reminder_channel          object
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome       object
dtype: object

Les dates sont stockées en texte au format mois/jour/année (2/6/2025) alors que le dictionnaire annonce l'ISO (2025-04-10). Conséquence : conversion obligatoire avec pd.to_datetime avant toute analyse par mois ou jour de semaine. 

In [15]:
df.duplicated().sum()

np.int64(0)

 0 doublon : aucune suppression nécessaire.

In [16]:
df.describe()

,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


previous_no_shows : médiane 0 mais moyenne 0,54 (max 5) → la plupart des patients ne manquent jamais, mais une minorité récidive beaucoup. Distance (0,5–45 km) et délai de réservation (0–60 j) très variables

In [18]:
df['appointment_outcome'].value_counts()

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

Trois issues : No-Show (2 423), Attended (2 314), Cancelled (263). Taux de no-show : 48,5 % de tous les RDV, 51,2 % des RDV non annulés. Je retiens 48,5 % : il mesure la part des absences sur l'ensemble des rendez-vous réservés, ce qui reflète la perte globale subie par l'activité de réservation de la clinique. Une annulation n'est pas un no-show : le patient prévient et libère le créneau ; le no-show laisse le créneau vide et fait perdre de l'argent.

In [19]:
df[df['previous_no_shows'] > df['previous_appointments']]

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome


In [20]:
df['booking_date'].head() 

0      2/6/2025
1     2/25/2026
2    11/16/2025
3     7/18/2025
4      7/9/2025
Name: booking_date, dtype: object

previous_no_shows : médiane 0 mais moyenne 0,54 (max 5) → la plupart des patients ne manquent jamais, mais une minorité récidive beaucoup. Distance (0,5–45 km) et délai de réservation (0–60 j) très variables

In [22]:
df[df['reminder_channel'].isna()]['reminder_sent'].value_counts()

reminder_sent
No    1366
Name: count, dtype: int64

In [23]:
no_show = 2423   
total = len(df)
sans_annules = len(df[df['appointment_outcome'] != 'Cancelled'])
print(no_show / total, no_show / sans_annules)

0.4846 0.5115051720498206


In [24]:
df.groupby('reminder_sent')['appointment_outcome'].value_counts(normalize=True)


reminder_sent  appointment_outcome
No             No-Show                0.513909
               Attended               0.426794
               Cancelled              0.059297
Yes            Attended               0.476335
               No-Show                0.473583
               Cancelled              0.050083
Name: proportion, dtype: float64

Sans rappel envoyé, la part de No-Show est nettement plus élevée : le rappel semble protecteur.

In [25]:
df.groupby('appointment_day')['appointment_outcome'].value_counts(normalize=True)

appointment_day  appointment_outcome
Friday           Attended               0.490385
                 No-Show                0.465659
                 Cancelled              0.043956
Monday           No-Show                0.496434
                 Attended               0.437946
                 Cancelled              0.065621
Saturday         Attended               0.475138
                 No-Show                0.468232
                 Cancelled              0.056630
Sunday           No-Show                0.504749
                 Attended               0.450475
                 Cancelled              0.044776
Thursday         No-Show                0.489766
                 Attended               0.451754
                 Cancelled              0.058480
Tuesday          Attended               0.490566
                 No-Show                0.470247
                 Cancelled              0.039187
Wednesday        No-Show                0.496608
                 Attended       

Le samedi enregistre le taux d'absentéisme le plus élevé (jour d'ouverture courte, 9 h–14 h), et le créneau du soir dépasse matin et après-midi. 

In [26]:
df.groupby('appointment_time')['appointment_outcome'].value_counts(normalize=True)

appointment_time  appointment_outcome
Afternoon         No-Show                0.483763
                  Attended               0.461318
                  Cancelled              0.054919
Evening           No-Show                0.497791
                  Attended               0.447717
                  Cancelled              0.054492
Morning           No-Show                0.481365
                  Attended               0.468792
                  Cancelled              0.049843
Name: proportion, dtype: float64

Le samedi enregistre le taux d'absentéisme le plus élevé (jour d'ouverture courte, 9 h–14 h), et le créneau du soir dépasse matin et après-midi. 

In [27]:
df.groupby('previous_no_shows')['appointment_outcome'].value_counts(normalize=True)

previous_no_shows  appointment_outcome
0                  Attended               0.504622
                   No-Show                0.435125
                   Cancelled              0.060253
1                  No-Show                0.534884
                   Attended               0.422481
                   Cancelled              0.042636
2                  No-Show                0.593607
                   Attended               0.363014
                   Cancelled              0.043379
3                  No-Show                0.679487
                   Attended               0.294872
                   Cancelled              0.025641
4                  No-Show                0.666667
                   Attended               0.333333
5                  No-Show                1.000000
Name: proportion, dtype: float64

Plus un patient a manqué de rendez-vous par le passé, plus il risque de manquer le suivant : le comportement passé est le signal le plus fort observé.

In [28]:
# distance et attente : compare ceux qui viennent vs ceux qui ratent
df.groupby('appointment_outcome')[['distance_to_clinic_km','waiting_time_minutes','booking_lead_days']].mean()

,distance_to_clinic_km,waiting_time_minutes,booking_lead_days
appointment_outcome,,,
Attended,9.668703,24.289141,24.521175
Cancelled,10.111583,23.214559,29.631179
No-Show,10.531481,24.200754,34.526620


Les No-Show habitent un peu plus loin (10,5 km vs 9,7) et réservent beaucoup plus à l'avance (34,5 j vs 24,5) → le long délai est un facteur fort, la distance un facteur modéré. En revanche, l'attente est quasi identique (24,2 vs 24,3) : hypothèse « attente → no-show » écartée. 

## Variables clés liées au no-show (mes hypothèses, classées)

1. **previous_no_shows** — plus un patient a manqué par le passé, plus son taux
   de no-show augmente : le comportement passé est le signal le plus fort observé.
2. **reminder_sent** — sans rappel envoyé, la part de No-Show est nettement plus
   élevée : le rappel semble protecteur.
3. **booking_lead_days** — les No-Show réservent ~34,5 j à l'avance contre ~24,5 j
   pour les Attended : plus le délai est long, plus on « oublie ».
4. **appointment_day** — le samedi enregistre le taux d'absentéisme le plus élevé.
5. **appointment_time** — le créneau du soir dépasse matin et après-midi.
6. **distance_to_clinic_km** — les No-Show habitent un peu plus loin
   (10,5 km vs 9,7 km) : effet modéré mais réel.

**Variable testée puis écartée :**
- **waiting_time_minutes** — attente quasi identique entre No-Show et Attended
  (24,2 vs 24,3 min) : hypothèse « l'attente pousse à l'absence » non confirmée.

In [ ]:
##Business questions

Q1. Les rappels de rendez-vous réduisent-ils réellement le taux d'absentéisme ?
Enjeu : le rappel a un coût ; mesurer son effet réel et choisir le canal le plus efficace.

Q2. Quels jours de semaine enregistrent le taux d'absentéisme le plus élevé ?
Enjeu : adapter les plannings et anticiper les jours à risque.

Q3. Le créneau horaire (matin / après-midi / soir) influence-t-il le risque d'absence ?
Enjeu : mieux organiser la répartition des créneaux.

Q4. L'éloignement entre le domicile du patient et la clinique est-il un facteur d'absentéisme ?
Enjeu : prévoir un accompagnement renforcé pour les patients éloignés.

Q5. Un long délai entre la réservation et le rendez-vous augmente-t-il le risque d'absence ?
Enjeu : déterminer le bon moment pour envoyer les rappels (piqûre de rappel pour les RDV réservés tôt).

Q6. Les patients ayant déjà manqué un rendez-vous manquent-ils davantage que les autres ?
Enjeu : détecter à l'avance les « récidivistes » pour leur offrir un suivi prioritaire.



### KPI 

KPI	Formule	Question	Pourquoi
Écart de taux de no-show avec/sans rappel	No-Show ÷ RDV dans chaque groupe (rappel envoyé / non envoyé)	Q1	mesure l'effet direct du rappel
Taux de no-show par créneau (jour × horaire)	No-Show ÷ RDV pour chaque jour et chaque créneau	Q2 + Q3	identifie les moments à risque
Taux de no-show par tranche de distance	No-Show ÷ RDV par tranche (< 5 / 5-15 / > 15 km)	Q4	teste l'effet de l'éloignement
Taux de no-show par tranche de délai	No-Show ÷ RDV par tranche (< 7 / 8-30 / > 30 j)	Q5	teste l'effet « plus c'est loin, plus on oublie »
Taux de no-show des récidivistes vs jamais manqué	No-Show ÷ RDV chez previous_no_shows ≥ 1, comparé à = 0	Q6	le signal le plus fort observé


## Initial analysis approach

Ma démarche pour les semaines suivantes, en 4 phases :

1. **Nettoyage & préparation** : convertir `booking_date` et `appointment_date`
   avec `pd.to_datetime` ; trancher le traitement des manquants (distance, attente) ;
   gérer `Cancelled` séparément des No-Show.
2. **Calcul des KPI** : implémenter les 5 KPI définis (groupby pandas) et vérifier
   chaque chiffre avant de le publier.
3. **Visualisation** : construire des visuels lisibles (taux de no-show par créneau,
   par tranche de distance/délai, récidivistes) dans Power BI ou matplotlib.
4. **Lecture & recommandations** : transformer les constats en actions pour la
   clinique (rappels ciblés, plannings, suivi des récidivistes).

In [ ]:
## Assumptions, limitations, risks

- ✔️ **Données fictives** : les patterns observés peuvent ne pas refléter la réalité
  d'une vraie clinique → conclusions à valider sur données réelles.
- ✔️ **Manquants documentés** : `waiting_time_minutes` (60) et `distance_to_clinic_km`
  (90) contiennent des valeurs absentes annoncées par le dictionnaire → un choix de
  traitement (conserver/immer) pourra influencer les résultats.
- + **Hypothèse de périmètre** : j'exclus `Cancelled` du calcul principal du taux de
  no-show (un annulé libère le créneau) → ce choix est documenté et pourra être revu.
- + **Corrélation ≠ causalité** : le lien rappel → moins d'absences est une association
  observée, pas une preuve d'effet.
- + **Périmètre adultes uniquement** (dictionnaire) : résultats non généralisables à
  une patientèle pédiatrique.